# Annual Sales Forecast | FSU / DTO | Walk-Forward Model with Residual Correction

Forecasts first-year annual sales for new FSU and DTO stores, then learns from the out-of-fold
errors of earlier cohorts to correct biases that repeat year after year.

**Steps:** 1) data transformation, 2) feature selection and cleaning, 3) walk-forward backtest,
4) residual analysis, 5) residual-correction backtest, 6) clustering of remaining errors,
7) final model training, 8) model save, 9) inference.

Inputs are read live from the feature tables refreshed by the upstream pipeline on every run; nothing
is read from cached results. Plots and documentation tables are in `sales-forecast-analysis.ipynb`.

Helper code lives in the `sales_forecast/` package next to this notebook; tunable values are in
`sales_forecast/constants.py`. See `sales_forecast/README.md` for the full method documentation.

## Environment Setup
On Databricks, add the standard two cells first (`%run .../adh_init` and `__adh_init()`); the data
section then reads the live tables from `adh.conf.default_schema_qualified`.
Locally, `PACKAGE_DIR` points at the pinned Python packages (`.annual_packages` by default; override with
`SALES_PACKAGE_DIR`), and inputs fall back to the development extracts.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
PACKAGE_DIR = Path(os.environ.get("SALES_PACKAGE_DIR", ROOT / ".annual_packages"))
sys.path.insert(0, str(PACKAGE_DIR))
sys.path.insert(0, str(ROOT))
os.environ.setdefault("OMP_NUM_THREADS", "4")

import warnings

import numpy as np
import pandas as pd

from sales_forecast import clusters as K
from sales_forecast import constants as C
from sales_forecast import residuals as R
from sales_forecast.features import FeatureSelector
from sales_forecast.metrics import summarize
from sales_forecast.model import (
    SalesForecaster,
    walk_forward_backtest,
    walk_forward_base,
)
from sales_forecast.sources import load_inputs
from sales_forecast.transform import (
    build_store_attributes,
    engineer_features,
    prepare_location_attributes,
    prepare_sales_features,
    split_cohorts,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
try:
    display
except NameError:  # running outside Jupyter / Databricks
    display = print

C.OUTPUT_DIR.mkdir(exist_ok=True)
print(f"{C.AS_OF_DATE=} {C.INCLUDED_SUBTYPES=} {C.OUTPUT_DIR=}")

C.AS_OF_DATE=Timestamp('2026-10-01 00:00:00') C.INCLUDED_SUBTYPES=['FSU', 'DTO'] C.OUTPUT_DIR=WindowsPath('C:/Users/SuBae/OneDrive - Green Cabbage/Documents/sales/sales_forecast_results')


## 1. Data transformation
- Read the latest data. On Databricks: the `location_features`, `boundary_features`, and
  `hexbin_level_features` tables (same joins as `sales-forecast.ipynb`), plus the store-attribute table
  `LOCATION_ATTRIBUTES_TABLE`. Outside Databricks: the development extracts with the same columns.
- `AS_OF_DATE` defaults to today, so the newest cohort with a full first year is used automatically.
- Keep FSU and DTO only. MALL stores are excluded.
- Target = `12 × avg_monthly_sales` in $M, modelled in log space.
- Cohort = calendar year of the first anniversary (`open_date + 1 year`).
- Relocation: `is_relocated` from the attributes table. Predecessor = a closed "(OLD)" location
  with the same normalized name; only its age is used, since its sales are not available.
- Training (labeled) stores: existing, open more than one year at the as-of date, positive sales.
  Inference stores: pipeline stores plus existing stores without a full first year yet.

In [2]:
schema = adh.conf.default_schema_qualified if "adh" in globals() else C.FEATURE_SCHEMA
sales_features_raw, location_attributes_raw = load_inputs(spark=globals().get("spark"), schema=schema)
sales_features = prepare_sales_features(sales_features_raw)
location_attributes = prepare_location_attributes(location_attributes_raw)
store_attributes = build_store_attributes(location_attributes)
print(f"{sales_features.shape=} {location_attributes.shape=}")

df_all = engineer_features(sales_features, store_attributes)
labeled, inference = split_cohorts(df_all)
print(f"{df_all.shape=} {labeled.shape=} {inference.shape=}")
assert labeled["location_num"].is_unique and inference["location_num"].is_unique
assert not set(labeled["location_num"]) & set(inference["location_num"])

geography_missing = inference[["region", "dma", "submarket"]].eq("__MISSING__").mean().rename("missing_share_inference")
display(geography_missing.round(3).to_frame())  # region is imputed from DMA/state; missing DMA/submarket fall back to neutral encodings

source='files' sales_features.shape=(3432, 76) location_attributes.shape=(8103, 16)
sales_features.shape=(3432, 76) location_attributes.shape=(8103, 16)
df_all.shape=(3190, 116) labeled.shape=(1238, 116) inference.shape=(721, 116)


,missing_share_inference
region,0.001
dma,0.028
submarket,0.245


In [3]:
cohort_summary = labeled.groupby(["cohort", "subtype"])[C.target].agg(["count", "median", "mean", "std"]).unstack()
display(cohort_summary.round(2))

count        median        mean         std      
subtype   DTO    FSU    DTO   FSU   DTO   FSU   DTO   FSU
cohort                                                   
2016      NaN   26.0    NaN  3.83   NaN  3.81   NaN  0.97
2017      1.0   94.0   3.60  4.39  3.60  4.48   NaN  1.32
2018      1.0  112.0   3.98  4.68  3.98  4.77   NaN  1.28
2019      5.0  106.0   5.04  5.26  5.04  5.43  1.20  1.53
2020      3.0  115.0   5.50  5.66  5.37  5.89  0.31  1.42
2021      2.0  100.0   6.61  6.14  6.61  6.40  1.60  1.71
2022      3.0  132.0   5.13  6.60  5.05  6.86  0.62  1.92
2023      7.0  116.0   7.42  7.21  6.75  7.22  1.86  1.94
2024     13.0  142.0   7.64  7.67  7.85  7.87  1.91  2.24
2025     16.0  126.0   6.47  7.27  6.63  7.55  2.26  2.15
2026     13.0  105.0   6.27  7.26  7.32  7.60  3.82  2.25

In [4]:
relocation_summary = labeled.groupby(["is_relocated", "predecessor_found"])[C.target].agg(["count", "median"])
display(relocation_summary.round(2))
display(inference.groupby(["location_type", "cohort"]).size().rename("stores").reset_index())

count  median
is_relocated predecessor_found               
0            0                   1147    5.89
1            0                     41    7.66
             1                     50    8.73

,location_type,cohort,stores
0,existing,2026,70
1,existing,2027,99
2,pipeline,2027,84
3,pipeline,2028,182
4,pipeline,2029,193
5,pipeline,2030,78
6,pipeline,2031,9
7,pipeline,2032,4
8,pipeline,2041,1
9,pipeline,2042,1


## 2. Feature selection and cleaning
- Infinite values become missing. The tree models handle missing values natively.
- Region and state are encoded as integer codes learned on the training rows; unseen values become -1.
- A feature is dropped if more than `MAX_NULL_SHARE` of training rows are missing, or if it is constant.
- Physical store attributes (building size, lanes, occupancy) are **not** used: they are mostly blank
  for pipeline stores, and unopened stores show 0 lanes.
- Time-safe outcome features (time trend, spatially smoothed sales of nearby earlier stores, and shrunk
  state/DMA/submarket/region averages) are added inside the model. Training rows only see earlier cohorts.

In [5]:
selector = FeatureSelector().fit(labeled)
print(f"{len(C.add)=} candidate features, {len(selector.features_)=} kept")
display(selector.report_[~selector.report_["keep"]])

len(C.add)=70 candidate features, len(selector.features_)=70 kept


,null_share,n_unique,keep,reason


In [6]:
feature_profile = selector.transform(labeled).describe(percentiles=[0.25, 0.5, 0.75]).T
display(feature_profile.round(3))

,count,mean,std,min,25%,50%,75%,max
region_code,1238.0,2.713,1.655,0.000,1.000,3.000,4.000,5.000
state_code,1238.0,23.183,14.600,0.000,9.000,24.000,39.000,45.000
ssam_flag,1238.0,0.085,0.279,0.000,0.000,0.000,0.000,1.000
no_dt_flag,1238.0,0.985,0.120,0.000,1.000,1.000,1.000,1.000
is_dto,1238.0,0.052,0.222,0.000,0.000,0.000,0.000,1.000
month_of_opening,1238.0,6.581,3.551,1.000,3.000,7.000,10.000,12.000
density_classification_1,1238.0,6.405,0.535,3.000,6.000,6.000,7.000,7.000
boundary_5,1238.0,0.842,1.216,0.000,0.000,0.000,1.000,11.000
boundary_10,1238.0,2.129,2.696,0.000,0.000,1.000,3.000,22.000
boundary_15,1238.0,2.562,3.191,0.000,0.000,1.000,4.000,21.000


## 3. Walk-forward backtest (base model)
Each cohort from `FIRST_OOF_COHORT` to the latest is predicted by a model trained only on earlier cohorts.
These out-of-fold (OOF) predictions are the input for the residual analysis and correction.
Training-set residuals are not used: they are overfit and would hide real errors.

Always recomputed from the current data (about 5 minutes). The OOF table is written to `OUTPUT_DIR` as an
output for review only; it is never read back.

In [7]:
oof_cohorts = list(range(C.FIRST_OOF_COHORT, int(labeled["cohort"].max()) + 1))
oof = walk_forward_base(labeled, oof_cohorts)
oof.assign(location_num=labeled.loc[oof.index, "location_num"]).to_csv(C.OUTPUT_DIR / "oof_base_predictions.csv", index=False)
print(f"{oof.shape=}")

cohort=2018 len(train)=121 len(test)=113 base WAPE=0.1901
cohort=2019 len(train)=234 len(test)=111 base WAPE=0.1784
cohort=2020 len(train)=345 len(test)=118 base WAPE=0.1597
cohort=2021 len(train)=463 len(test)=102 base WAPE=0.1506
cohort=2022 len(train)=565 len(test)=135 base WAPE=0.1908
cohort=2023 len(train)=700 len(test)=123 base WAPE=0.1916
cohort=2024 len(train)=823 len(test)=155 base WAPE=0.1816
cohort=2025 len(train)=978 len(test)=142 base WAPE=0.2010
cohort=2026 len(train)=1120 len(test)=118 base WAPE=0.1949
oof.shape=(1117, 4)


## 4. Residual analysis: what do earlier errors say about later cohorts?
`residual = log(actual) - log(forecast)`, out-of-fold. Positive = under-forecast.
`centred` removes each cohort's median residual, so a group effect is not confused with that year's overall level.

- **A. Cohort bias:** is the overall level error of one year repeated the next year?
- **B. Group lag effects:** does a group's error in cohort Y-1, Y-2, Y-3 predict cohort Y?
- **C. Spatial lag effects:** does the error of nearby stores from an earlier cohort predict a new store's error?
- **D. Feature bins:** are some segments (e.g. late-year openings) consistently under- or over-forecast?

In [8]:
residual_frame = labeled.loc[oof.index].assign(residual=labeled.loc[oof.index, C.log_target] - oof["base"])
residual_frame = R.add_centred(residual_frame)
print(f"residual sd={residual_frame['residual'].std():.3f}")

bias = R.cohort_bias(residual_frame)
display((bias * 100).round(1).T)
print(f"lag-1 autocorrelation of cohort bias: {bias['median_residual'].autocorr(1):.2f}")

residual sd=0.233


cohort,2018,2019,2020,2021,2022,2023,2024,2025,2026
median_residual,-2.6,4.1,-2.0,-3.3,-6.3,4.3,4.9,-9.9,-1.8
previous_cohort,NaN,-2.6,4.1,-2.0,-3.3,-6.3,4.3,4.9,-9.9


lag-1 autocorrelation of cohort bias: -0.20


In [9]:
group_lags = R.group_lag_correlation(
    residual_frame,
    ["region", "state", "dma", "subtype", "ssam_flag", "is_relocated"],
)
display(group_lags.round(2))

corr             pairs            
lag              1     2     3     1     2     3
group                                           
dma          -0.13 -0.24 -0.08  51.0  47.0  33.0
is_relocated  0.73  0.85  0.76  16.0  14.0  12.0
region        0.10 -0.01 -0.18  48.0  42.0  36.0
ssam_flag     0.38  0.34  0.20  14.0  12.0  10.0
state        -0.15 -0.09 -0.05  91.0  80.0  63.0
subtype       0.57  0.21  0.28  13.0  11.0  10.0

In [10]:
spatial_lags = R.spatial_lag_correlation(residual_frame)
display(spatial_lags.round(2))

corr                  n              
lag              1     2     3      1      2      3
bandwidth_km                                       
10            0.25 -0.06  0.24  123.0   84.0   81.0
25            0.11 -0.02  0.02  373.0  306.0  260.0
60            0.05 -0.04  0.00  644.0  554.0  474.0

In [11]:
bin_features = [
    "is_relocated", "predecessor_found", "ssam_flag", "is_dto", "month_of_opening", "bigbox_1",
    "log_med_household_income_1", "neighbor_closest_vs_mean", "log_max_average_daily_traffic_1",
    "boundary_5", "boundary_10", "cannibalization_score", "log_total_population_1",
    "log_avg_monthly_sales_mean_1year", "density_classification_1", "latitude", "longitude",
]
bin_persistence = R.feature_bin_persistence(residual_frame, bin_features)
display(bin_persistence.round(3))

,feature,wf_corr,lowest_bin_pct,highest_bin_pct,sign_consistency
0,is_relocated,0.915,-1.462,12.194,1.000
1,predecessor_found,0.900,-1.018,13.637,1.000
2,month_of_opening,0.412,-0.034,-5.975,0.778
3,is_dto,0.357,0.169,-9.240,0.889
4,ssam_flag,0.329,0.226,-6.277,0.778
5,bigbox_1,0.317,-1.812,3.379,0.889
6,boundary_5,0.298,3.505,-5.455,0.889
7,log_med_household_income_1,0.281,-3.212,-2.021,0.889
8,cannibalization_score,0.147,2.825,-4.757,0.889
9,neighbor_closest_vs_mean,0.139,3.466,0.904,0.778


In [12]:
residual_frame["opening_quarter"] = (residual_frame["month_of_opening"] - 1) // 3 + 1
segment_table = (
    residual_frame.melt(id_vars=["cohort", "centred"], value_vars=["is_relocated", "ssam_flag", "is_dto", "opening_quarter"])
    .groupby(["variable", "value", "cohort"])["centred"].mean().unstack()
)
display((segment_table * 100).round(1))

cohort                 2018  2019  2020  2021  2022  2023  2024  2025  2026
variable        value                                                      
is_dto          0      -0.3   1.5  -0.9   1.3   0.1  -0.6  -1.2   0.7   1.7
                1     -15.5  -3.2  -5.6   1.4 -38.2  -7.7  -6.8 -13.1  -5.5
is_relocated    0      -1.2   1.1  -1.6   0.3  -2.0  -3.3  -3.2  -1.8  -0.4
                1      14.6   3.1  17.7  12.6  17.7  15.7  10.9   7.0  14.7
opening_quarter 1       2.3  -4.6  -0.3   2.2   2.4   0.0   1.0  11.6  -1.6
                2      -0.8   0.3   2.9   4.3   7.9   3.3   1.4  -6.5   5.0
                3      -3.3   9.0  -3.3  -2.2   1.5   2.8  -0.9   4.6  -2.7
                4      -0.6   3.6  -3.3  -0.4  -8.8  -8.0  -5.4  -5.1   NaN
ssam_flag       0       0.7   2.4  -0.2   1.9  -1.1   0.9  -1.5  -0.9   1.0
                1     -11.8 -13.1  -6.9  -3.2   3.1 -12.8  -2.5   NaN  -7.6

**Reading the results** (see section 5 for the effect on accuracy):
- Errors that **repeat**: relocated stores (under-forecast), SSAM and DTO stores (over-forecast),
  and late-year openings (over-forecast). These become the `RESIDUAL_FACTORS` correction.
- Errors that **do not repeat**: overall cohort level, region, state, and DMA. Their lag correlations are near zero
  or negative, so an "under-forecast last year, raise this year" rule by geography adds noise.
  Nearby-store sales already capture the local price and demand level.
- Spatial lag effects are weak (only within about 10 km at a one-cohort lag) and did not improve accuracy.

## 5. Residual-correction backtest
For each scored cohort Y, the `ResidualCorrector` is fit only on OOF residuals of cohorts < Y:
1. **Factor offsets:** ridge regression of the centred residual on relocated / SSAM / DTO / opening quarter / log cannibalization (alpha = `RESIDUAL_RIDGE_ALPHA`).
2. **Level anchor:** move the cohort's median forecast `LEVEL_ANCHOR_STRENGTH` of the way toward the latest cohort's actual median.

In [13]:
backtest, backtest_frame = walk_forward_backtest(labeled, oof, C.BACKTEST_COHORTS)
backtest_table = backtest.pivot(index="model", columns="cohort", values=["WAPE", "MAPE"])
for metric in ["WAPE", "MAPE"]:
    backtest_table[(metric, "avg")] = backtest_table[metric].mean(axis=1)
display((backtest_table.sort_index(axis=1) * 100).round(1))
backtest.to_csv(C.OUTPUT_DIR / "backtest_metrics.csv", index=False)

MAPE                                            WAPE                                          
cohort     2020  2021  2022  2023  2024  2025  2026   avg  2020  2021  2022  2023  2024  2025  2026   avg
model                                                                                                    
base       17.0  15.5  20.7  19.5  18.4  23.5  20.2  19.3  16.0  15.1  19.1  19.2  18.2  20.1  19.5  18.1
corrected  16.2  14.4  17.7  18.5  17.0  20.5  18.5  17.5  16.4  15.3  17.9  19.0  17.3  18.1  18.6  17.5

In [14]:
scored = backtest_frame[backtest_frame["cohort"].isin(C.BACKTEST_COHORTS)]
segment_rows = []
for column in ["subtype", "region", "is_relocated", "ssam_flag"]:
    for value, part in scored.groupby(column):
        segment_rows.append(dict(segment=column, value=value, **summarize(part[C.target], np.exp(part["log_corrected"]))))
display(pd.DataFrame(segment_rows).round(3))

,segment,value,n,WAPE,MAPE,median_APE,bias
0,subtype,DTO,57,0.216,0.222,0.178,-0.022
1,subtype,FSU,836,0.174,0.174,0.146,-0.055
2,region,Atlantic Region,112,0.163,0.159,0.129,-0.058
3,region,Midwest Region,150,0.170,0.176,0.147,-0.020
4,region,Northeast Region,125,0.177,0.174,0.165,-0.062
5,region,Southeast Region,167,0.176,0.189,0.139,-0.033
6,region,Southwest Region,181,0.168,0.162,0.146,-0.057
7,region,West Region,158,0.198,0.195,0.161,-0.086
8,is_relocated,0,815,0.178,0.178,0.150,-0.049
9,is_relocated,1,78,0.162,0.165,0.146,-0.091


## 6. Clustering of remaining errors
After the correction, which kinds of stores are still under- or over-forecast? Residuals are taken from the
walk-forward backtest (cohorts 2020 onward), so they are out-of-sample.
- **K-means** on standardized location characteristics (`CLUSTER_FEATURES`). A cluster effect is only useful if it
  **persists**: its mean error in earlier cohorts must predict its error in later cohorts (walk-forward correlation).
- **Under vs over profile:** feature z-scores of the 20% most under-forecast vs the 20% most over-forecast stores.
- **Residual tree:** a shallow regression tree gives readable rules; its walk-forward correlation shows whether the rules generalize.

In [15]:
remaining = backtest_frame[backtest_frame["cohort"].isin(C.BACKTEST_COHORTS)].copy()
remaining["residual"] = remaining[C.log_target] - remaining["log_corrected"]
remaining = R.add_centred(remaining)

display(K.k_sweep(remaining).round(3))
cluster_labels, cluster_centroids = K.fit_clusters(remaining)
remaining["cluster"] = cluster_labels
display(K.cluster_summary(remaining, cluster_labels).round(3))
display(cluster_centroids.round(2))
print(K.cluster_persistence(remaining, cluster_labels))

,k,between_cluster_sd_pct,walk_forward_corr,pairs
0,4,1.798,0.129,24
1,6,2.379,0.126,35
2,8,1.863,-0.049,41
3,10,2.265,-0.128,48


,stores,mean_residual_pct,under_forecast_share,over_forecast_share,median_sales_m
cluster,,,,,
0,92,-3.354,0.130,0.207,6.083
1,170,1.675,0.300,0.259,7.323
2,74,-2.260,0.203,0.284,6.169
3,193,0.806,0.254,0.207,6.792
4,145,2.606,0.303,0.228,6.922
5,219,-1.550,0.224,0.242,7.040


,0,1,2,3,4,5
log_total_population_1,-0.55,1.08,-1.98,0.19,-0.38,0.15
log_total_households_per_sq_mile_1,-0.54,1.07,-1.95,0.09,-0.33,0.20
log_med_household_income_1,-0.48,0.35,-0.77,0.70,0.12,-0.50
worker_resident_ratio,-0.05,-0.29,1.36,-0.97,0.90,0.04
daytime_resident_ratio,-0.02,-0.34,1.60,-0.84,0.89,-0.12
log_max_average_daily_traffic_1,-0.40,0.52,-0.33,-0.29,0.11,0.06
log_avg_monthly_sales_mean_1year,-0.13,0.24,-0.05,-0.05,-0.49,0.26
neighbor_closest_vs_mean,-0.19,-0.22,0.01,0.35,-0.33,0.15
neighbor_spread,-0.04,0.35,-0.37,0.11,0.57,-0.61
cannibalization_score,-0.53,1.18,-0.67,0.06,-0.11,-0.45


{'walk_forward_corr': np.float64(0.12624084647660283), 'pairs': 35}


In [16]:
under_over = K.under_over_profile(remaining)
display(under_over.round(2))
tree_rules, tree_corr = K.residual_tree(remaining)
print(f"residual tree walk-forward corr={tree_corr:.3f}")
print(tree_rules)

,under_forecast_z,over_forecast_z,gap
neighbor_closest_vs_mean,-0.09,0.10,-0.19
close_highway_min,-0.17,-0.03,-0.14
cannibalization_score,0.00,0.09,-0.08
log_max_average_daily_traffic_1,-0.01,0.05,-0.06
log_avg_monthly_sales_mean_1year,0.03,0.06,-0.03
highway_5,-0.01,0.01,-0.03
avg_population_growth_1,-0.08,-0.05,-0.03
worker_resident_ratio,-0.02,0.01,-0.02
neighbor_spread,-0.01,-0.01,0.00
daytime_resident_ratio,-0.00,-0.03,0.02


residual tree walk-forward corr=0.028
|--- neighbor_closest_vs_mean <= -0.04
|   |--- close_highway_min <= 8.18
|   |   |--- log_med_household_income_1 <= 11.35
|   |   |   |--- value: [0.12]
|   |   |--- log_med_household_income_1 >  11.35
|   |   |   |--- value: [0.05]
|   |--- close_highway_min >  8.18
|   |   |--- value: [-0.02]
|--- neighbor_closest_vs_mean >  -0.04
|   |--- neighbor_closest_vs_mean <= 0.39
|   |   |--- cannibalization_score <= 20.50
|   |   |   |--- value: [-0.02]
|   |   |--- cannibalization_score >  20.50
|   |   |   |--- value: [-0.09]
|   |--- neighbor_closest_vs_mean >  0.39
|   |   |--- value: [0.05]



**Reading the results:**
- K-means clusters of location types persist only weakly (walk-forward correlation about 0.1 or lower, depending on k),
  and their mean errors are small (a few percent). Clusters are kept as diagnostics, not as a correction.
- Before cannibalization was added to the correction, it was the clearest difference between over- and under-forecast
  stores: over-forecast stores had many more same-brand stores nearby (+0.45 SD). `log_cannibalization` is now in
  `RESIDUAL_FACTORS`. It lowered the 2020-2026 walk-forward MAPE by about 0.6 points and WAPE by about 0.1 point; it was
  chosen after looking at these cohorts, so treat the gain as optimistic.
- What remains after the correction is small (all profile gaps below 0.2 SD), and the residual tree does not generalize
  (walk-forward correlation about 0.03). The remaining error is store-specific rather than a segment pattern.

## 7. Train final model
The base ensemble is retrained on **all** labeled stores. The residual corrector is fit on all OOF
residuals (cohorts `FIRST_OOF_COHORT` onward), so it uses every year of error history.

In [17]:
artifacts = dict(
    oof=oof.assign(location_num=labeled.loc[oof.index, "location_num"]),
    backtest=backtest,
    backtest_frame=backtest_frame,
    residual_frame=residual_frame,
    remaining=remaining,
    cluster_centroids=cluster_centroids,
    group_lags=group_lags,
    spatial_lags=spatial_lags,
    bin_persistence=bin_persistence,
    under_over=under_over,
)
forecaster = SalesForecaster().fit(labeled, oof, artifacts=artifacts)
print(f"{forecaster.trained_cohorts_=}")
print(f"latest cohort={forecaster.corrector_.latest_cohort_} median annual sales=${np.exp(forecaster.corrector_.latest_median_):.2f}M")
display((forecaster.corrector_.factor_offsets() * 100).round(1).rename("offset_pct_log").to_frame())

forecaster.trained_cohorts_=[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
latest cohort=2026 median annual sales=$7.02M


,offset_pct_log
is_relocated,11.7
ssam_flag,-9.1
is_dto,-5.4
open_q2,1.9
open_q3,-0.1
open_q4,-4.8
log_cannibalization,-3.5


## 8. Save model
The whole fitted object (feature selector, base models, trend, training reference rows, corrector, and this
run's backtest artifacts for the analysis notebook) is pickled.
Reload it with `SalesForecaster.load()`; the reloaded model must give identical predictions.

In [18]:
model_path = forecaster.save(C.MODEL_PATH)
reloaded = SalesForecaster.load(model_path)
sample = inference.head(20)
assert np.allclose(forecaster.predict(sample)["forecast_annual_sales"], reloaded.predict(sample)["forecast_annual_sales"])
print(f"saved {model_path} ({model_path.stat().st_size / 1e6:.1f} MB)")

saved C:\Users\SuBae\OneDrive - Green Cabbage\Documents\sales\sales_forecast_results\sales_forecast_model.pkl (6.0 MB)


## 9. Inference
Forecast pipeline stores and existing stores that do not yet have a full first year.
`forecast_base_annual_sales` is the uncorrected ensemble; `forecast_annual_sales` includes the residual correction.
For cohorts beyond the latest observed one, the level anchor assumes `LEVEL_ANNUAL_GROWTH` per year (0 by default).

In [19]:
forecast = reloaded.predict(inference)
forecast_path = C.OUTPUT_DIR / "inference_forecasts.csv"
forecast.to_csv(forecast_path, index=False)
print(f"{forecast.shape=} -> {forecast_path}")

forecast_summary = forecast.groupby(["location_type", "cohort"]).agg(
    stores=("location_num", "size"),
    median_forecast_m=("forecast_annual_sales", lambda s: s.median() / 1e6),
    total_forecast_m=("forecast_annual_sales", lambda s: s.sum() / 1e6),
)
display(forecast_summary.round(2))
display(forecast.head(10))

forecast.shape=(721, 14) -> C:\Users\SuBae\OneDrive - Green Cabbage\Documents\sales\sales_forecast_results\inference_forecasts.csv


stores  median_forecast_m  total_forecast_m
location_type cohort                                             
existing      2026        70               6.52            462.09
              2027        99               7.19            728.56
pipeline      2027        84               6.87            599.79
              2028       182               7.25           1345.45
              2029       193               7.28           1427.76
              2030        78               7.30            582.53
              2031         9               7.73             74.78
              2032         4               6.60             25.86
              2041         1               6.07              6.07
              2042         1               6.48              6.48

,location_num,location_name,location_type,subtype,state,dma,region,open_date,cohort,is_relocated,forecast_base_annual_sales,forecast_annual_sales,forecast_avg_monthly_sales,open_date_placeholder
0,06980,Nestles Rd. & Los Coches Rd. FSU,pipeline,FSU,CA,"Monterey-Salinas, CA",West Region,2029-06-07,2030,0,8.905526e+06,8.728337e+06,727361.391273,False
1,06989,Ogden Rd. & Hwy 550 FSU,pipeline,FSU,CO,"Grand Junction-Montrose, CO",West Region,2029-06-07,2030,0,7.947726e+06,7.789594e+06,649132.831156,False
2,06988,Ponca City FSU,pipeline,FSU,OK,"Oklahoma City, OK",Southwest Region,2028-08-03,2029,0,6.982616e+06,6.722737e+06,560228.121351,False
3,06990,Mineral Wells FSU,pipeline,FSU,TX,"Dallas-Ft. Worth, TX",Southwest Region,2028-08-03,2029,0,6.405014e+06,6.166632e+06,513885.999789,False
4,06987,Fishermans Wharf,pipeline,FSU,CA,"San Francisco-Oakland-San Jose, CA",West Region,2029-08-03,2030,0,1.098869e+07,1.056181e+07,880150.779888,False
5,06984,Houston Downtown Inline,pipeline,FSU,TX,"Houston, TX",Southwest Region,2029-08-03,2030,0,7.079462e+06,5.911705e+06,492642.122867,False
6,04183,Old Hickory Blvd. (TN) FSU,pipeline,FSU,TN,"Nashville, TN",Atlantic Region,2029-11-15,2030,0,7.163284e+06,5.897541e+06,491461.776929,False
7,06975,12 Mile & Mound (MI) FSU,pipeline,FSU,MI,"Detroit, MI",Midwest Region,2028-09-21,2029,0,7.784903e+06,6.847094e+06,570591.144415,False
8,06974,I-5 & Euclid FSU,pipeline,FSU,CA,"Los Angeles, CA",West Region,2030-03-07,2031,0,8.814421e+06,7.582500e+06,631875.001656,False
9,06978,Radford FSU,pipeline,FSU,VA,"Roanoke-Lynchburg, VA",Atlantic Region,2028-08-31,2029,0,7.331884e+06,6.888585e+06,574048.729122,False


In [20]:
latest_actual = labeled.loc[labeled["cohort"] == labeled["cohort"].max(), C.target]
print(f"latest cohort actual: median=${latest_actual.median():.2f}M mean=${latest_actual.mean():.2f}M")
print(f"pipeline forecast:    median=${forecast.loc[forecast['location_type'] == 'pipeline', 'forecast_annual_sales'].median() / 1e6:.2f}M")
assert forecast["forecast_annual_sales"].between(0.5e6, 40e6).all(), "forecast outside plausible range"

latest cohort actual: median=$7.02M mean=$7.57M
pipeline forecast:    median=$7.22M
